# 3 — Snowflake Cortex AI
## "Built-in AI for unstructured clinical data"

Structured variant tables are the easy half. The other half — pathology narratives, clinical notes, trial protocols, literature — holds facts that live in prose, not columns.

Cortex gives you LLM functions as **SQL functions**. No endpoint to stand up, no API key to rotate, no data leaving the account, and the same role-based access control as every other object.

We have 2,000 synthetic pathology narratives to work with.

In [ ]:
USE WAREHOUSE GUARDANT_DEMO_WH;
USE SCHEMA DEMO.GUARDANT_DEMO;

SELECT report_id, specimen_id, report_date, pathologist,
       LENGTH(report_text) AS chars
FROM PATHOLOGY_REPORTS
LIMIT 5;

### What we are working with

Deliberately narrative and inconsistent, the way dictated reports actually are. The facts are in the prose. This is why regex is the wrong tool.

In [ ]:
SELECT report_text
FROM PATHOLOGY_REPORTS
WHERE report_id = (SELECT MIN(report_id) FROM PATHOLOGY_REPORTS);

### AI_EXTRACT — prose to columns

Ask questions in plain English, get back structured answers. One SQL function, no prompt engineering scaffolding.

This turns a free-text corpus into something you can join and aggregate.

In [ ]:
SELECT
    report_id,
    specimen_id,
    AI_EXTRACT(
        text => report_text,
        responseFormat => {
            'cancer_type':       'What is the primary cancer type?',
            'gene':              'Which gene had the reported alteration?',
            'vaf_percent':       'What was the variant allele frequency, as a number?',
            'targetable':        'Is a targeted therapy indicated? Answer yes or no.',
            'resistance':        'Does the report suggest a resistance mechanism? Answer yes or no.'
        }
    ) AS extracted
FROM PATHOLOGY_REPORTS
LIMIT 10;

Flatten the extraction into real columns, and it joins to the structured tables like anything else.

In [ ]:
WITH extracted AS (
    SELECT
        r.report_id,
        r.specimen_id,
        AI_EXTRACT(
            text => r.report_text,
            responseFormat => {
                'gene':       'Which gene had the reported alteration?',
                'targetable': 'Is a targeted therapy indicated? Answer only yes or no.'
            }
        ) AS e
    FROM PATHOLOGY_REPORTS r
    LIMIT 50
)
SELECT
    x.report_id,
    x.specimen_id,
    x.e:response:gene::VARCHAR       AS gene_from_narrative,
    LOWER(x.e:response:targetable::VARCHAR) AS targetable_from_narrative,
    p.primary_cancer_type,
    s.tumor_fraction
FROM extracted x
JOIN SPECIMENS s ON s.specimen_id = x.specimen_id
JOIN PATIENTS  p ON p.patient_id  = s.patient_id;

### AI_CLASSIFY — triage at corpus scale

Route reports into the buckets a molecular tumour board actually cares about. Categories are supplied inline; there is no model to train.

In [ ]:
SELECT
    report_id,
    AI_CLASSIFY(
        report_text,
        ['actionable alteration found',
         'no actionable alteration',
         'resistance mechanism emerging',
         'no change from prior testing']
    ):labels[0]::VARCHAR AS triage_bucket
FROM PATHOLOGY_REPORTS
LIMIT 25;

Classify a slice and aggregate it — a corpus-level view of a text pile that was previously only readable one report at a time.

In [ ]:
WITH classified AS (
    SELECT
        AI_CLASSIFY(
            report_text,
            ['actionable alteration found',
             'no actionable alteration',
             'resistance mechanism emerging',
             'no change from prior testing']
        ):labels[0]::VARCHAR AS triage_bucket
    FROM PATHOLOGY_REPORTS
    LIMIT 200
)
SELECT triage_bucket,
       COUNT(*)                                           AS reports,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM classified
GROUP BY triage_bucket
ORDER BY reports DESC;

### AI_COMPLETE — generation with your own instructions

Where the task is not extraction or classification but summarisation or drafting, `AI_COMPLETE` takes a prompt and a model.

Here: a one-line clinician-facing summary per report.

In [ ]:
SELECT
    report_id,
    AI_COMPLETE(
        'claude-4-sonnet',
        CONCAT(
            'You are assisting a molecular tumour board. In ONE sentence of at most 30 words, ',
            'state the actionable finding and the recommended next step. No preamble.\n\n',
            report_text
        )
    ) AS board_summary
FROM PATHOLOGY_REPORTS
LIMIT 5;

### AI_AGG — reasoning across many rows at once

`AI_AGG` is the one with no laptop equivalent at all: it reasons over a whole *group* of text values, not row by row. Ask what the themes are across a cohort's reports.

In [ ]:
WITH cohort AS (
    SELECT r.report_text, p.primary_cancer_type
    FROM PATHOLOGY_REPORTS r
    JOIN SPECIMENS s ON s.specimen_id = r.specimen_id
    JOIN PATIENTS  p ON p.patient_id  = s.patient_id
    WHERE p.primary_cancer_type IN ('Non-Small Cell Lung', 'Colorectal')
    LIMIT 120
)
SELECT
    primary_cancer_type,
    COUNT(*) AS reports_reviewed,
    AI_AGG(
        report_text,
        'Across these pathology reports, summarise in 3 short bullets: the most common
         alterations mentioned, any recurring resistance patterns, and the dominant
         recommended next step. Be specific and concise.'
    ) AS cohort_themes
FROM cohort
GROUP BY primary_cancer_type;

### The same functions from Python

If the team prefers to stay in Snowpark, Cortex is available there too — so an AI step drops into an existing DataFrame pipeline rather than sitting beside it.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()

reports = session.table("PATHOLOGY_REPORTS").limit(8)

summarised = reports.select(
    F.col("REPORT_ID"),
    F.call_function(
        "AI_COMPLETE",
        F.lit("claude-4-sonnet"),
        F.concat(
            F.lit("In under 20 words, state the single key finding. No preamble.\n\n"),
            F.col("REPORT_TEXT"),
        ),
    ).alias("KEY_FINDING"),
)

summarised.to_pandas()

---

### What changed

| | Today | Here |
|---|---|---|
| Unstructured data | Read manually, or not at all | Queryable |
| AI on clinical text | External API, data leaves | In-account, governed |
| Access control | Separate system | Same roles as everything else |
| Structured + text together | Two pipelines | One SQL statement |

**Governance point worth making out loud:** every call above ran inside the Snowflake boundary. No report text was sent to a third-party endpoint.